In [1]:
#!/usr/bin/env python3
"""
Generate ChimeraX defattr file for B-factors from difference data.
Sums difference values by site and maps to sequential site numbering.
"""

import pandas as pd


In [2]:
def main():
    # Read the input files
    print("Reading input files...")
    diffs_df = pd.read_csv('../../results/summaries/tufted_duck_MHCII_binding.csv')

    # Filter to only rows where SA23 entry >= -3
    diffs_df = diffs_df[diffs_df['entry in SA23 cells'] >= -3]

    # Sum the difference values by site
    print("\nSumming difference values by site...")
    site_sums = diffs_df.groupby('site')['entry difference between noSA tufted duck MHCII and SA23 cells'].sum().reset_index()
    site_sums.columns = ['site', 'total_difference']

    # Generate the defattr file
    output_file = 'bfactors_entry_difference_H3_numbering.defattr'
    print(f"\nWriting defattr file to {output_file}...")
    with open(output_file, 'w') as f:
        # Write header
        f.write("# ChimeraX defattr file\n")
        f.write("# B-factors derived from summed difference values per site\n")
        f.write("#\n")
        f.write("attribute: difference\n")
        f.write("match mode: any\n")
        f.write("recipient: residues\n")
        f.write("\n")
        # Write data lines - handle non-numeric site values
        for _, row in site_sums.iterrows():
            site_value = str(row['site'])
            # Convert letters to uppercase for ChimeraX format (125a -> 125A)
            site_formatted = site_value.replace('a', 'A').replace('b', 'B').replace('c', 'C')
            bfactor = float(row['total_difference'])
            f.write(f"\t:{site_formatted}\t{bfactor:.6f}\n")

    print(f"\nDefattr file successfully created: {output_file}")
    print(f"Total residues with B-factors: {len(site_sums)}")
    print(f"\nB-factor statistics:")
    print(f"  Min: {site_sums['total_difference'].min():.6f}")
    print(f"  Max: {site_sums['total_difference'].max():.6f}")
    print(f"  Mean: {site_sums['total_difference'].mean():.6f}")
    print(f"  Median: {site_sums['total_difference'].median():.6f}")
    print("\nExample lines from defattr file:")
    print(site_sums.head(5).to_string(index=False))

if __name__ == "__main__":
    main()

Reading input files...

Summing difference values by site...

Writing defattr file to bfactors_entry_difference_H3_numbering.defattr...

Defattr file successfully created: bfactors_entry_difference_H3_numbering.defattr
Total residues with B-factors: 564

B-factor statistics:
  Min: -52.452000
  Max: 20.470000
  Mean: 1.490159
  Median: 0.992500

Example lines from defattr file:
site  total_difference
  -1          2.984130
  -2          1.141000
  -3          4.014150
  -4         -1.767890
  -5          0.045365
